# SeaDronesSee ConvNeXt QAT — Kaggle
Mỗi lần chạy đúng **1 epoch**, validation + benchmark 100 ảnh, lưu checkpoint vào `/kaggle/working`, sau đó tạo phiên bản mới của Kaggle checkpoint dataset để phiên sau resume. Bật **Settings → Accelerator → GPU** và **Internet** trước khi chạy.

In [ ]:
import os, shutil, subprocess, sys, datetime, json
from pathlib import Path
import torch
HAS_CUDA = torch.cuda.is_available()
print('CUDA available:', HAS_CUDA)
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
else:
    print('CPU mode: phù hợp để bootstrap/upload dataset; bật GPU trước khi train')

## 1. Clone và cài project
Internet phải được bật để clone GitHub.

In [ ]:
REPO = Path('/kaggle/working/EchteAI')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'kagglehub'], check=True)
print('Repository:', Path.cwd())

## 2. Khai báo Kaggle Dataset
Bạn chỉ cần nhập username Kaggle. Notebook tự tạo **Private Dataset** `seadronessee-compressed` nếu chưa tồn tại, tải dữ liệu từ Tübingen và upload lần đầu. Dataset checkpoint cũng được tự tạo private sau epoch đầu. Những phiên sau tự tải version mới nhất.

In [ ]:
import kagglehub
KAGGLE_USERNAME = 'YOUR_USERNAME'
SEADRONESSEE_DATASET = f'{KAGGLE_USERNAME}/seadronessee-compressed'
CHECKPOINT_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-checkpoints'
STAGE = 'fp32'  # fp32 hoặc qat
VARIANT = 'M3'
assert KAGGLE_USERNAME != 'YOUR_USERNAME', 'Hãy nhập username Kaggle của bạn'
print('Data:', SEADRONESSEE_DATASET)
print('Checkpoints:', CHECKPOINT_DATASET)
print('Next stage:', STAGE)

In [ ]:
def find_dataset_root(start):
    start = Path(start)
    candidates = [start, *[p.parent.parent for p in start.rglob('instances_train.json')]]
    for candidate in candidates:
        if (candidate/'annotations/instances_train.json').exists() and (candidate/'images/train').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy cấu trúc images/ + annotations/ trong {start}')

DATA_ROOT = None
try:
    DATA_DOWNLOAD = Path(kagglehub.dataset_download(SEADRONESSEE_DATASET))
    DATA_ROOT = find_dataset_root(DATA_DOWNLOAD)
    print('Using SeaDronesSee already stored on Kaggle')
except Exception as error:
    print('SeaDronesSee private dataset is missing or incomplete:', error)
    print('Bootstrapping it automatically from Tübingen...')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'rclone'], check=True)
    DAV_URL = 'https://cloud.cs.uni-tuebingen.de/public.php/dav/files/ZZxX65FGnQ8zjBP/'
    subprocess.run(['rclone', 'config', 'delete', 'seadrones'], check=False, capture_output=True)
    subprocess.run(['rclone', 'config', 'create', 'seadrones', 'webdav', 'url', DAV_URL, 'vendor', 'nextcloud'], check=True)
    DATA_ROOT = Path('/kaggle/working/seadronessee')
    subprocess.run([
        'rclone', 'copy', 'seadrones:Compressed Version', str(DATA_ROOT),
        '--progress', '--transfers', '2', '--checkers', '4', '--retries', '10',
    ], check=True)
    DATA_ROOT = find_dataset_root(DATA_ROOT)
    print('Uploading SeaDronesSee to the private Kaggle dataset...')
    kagglehub.dataset_upload(
        SEADRONESSEE_DATASET, str(DATA_ROOT),
        version_notes='SeaDronesSee compressed images and COCO annotations',
    )
    print('SeaDronesSee Kaggle dataset version uploaded')
print('Dataset root:', DATA_ROOT)
print('Train images:', len(list((DATA_ROOT/'images/train').glob('*'))))

## 3. Khôi phục checkpoint của phiên trước
Kaggle input là read-only, vì vậy checkpoint được copy sang `/kaggle/working/checkpoints`.

In [ ]:
OUTPUT = Path('/kaggle/working/checkpoints')
OUTPUT.mkdir(parents=True, exist_ok=True)
try:
    previous = Path(kagglehub.dataset_download(CHECKPOINT_DATASET))
    copied = 0
    for source in previous.rglob('*'):
        if source.is_file():
            shutil.copy2(source, OUTPUT/source.name)
            copied += 1
    print(f'Restored {copied} checkpoint/output files from {previous}')
except Exception as error:
    print('No previous checkpoint dataset; starting a new run:', error)

## 4. Tạo config runtime cho Kaggle

In [ ]:
import yaml
base = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(DATA_ROOT/'images/train'),
    'train_annotations': str(DATA_ROOT/'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT/'images/val'),
    'val_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT/'images/val'),
    'test_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
})
base['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(OUTPUT/'fp32_best.pt'), 'fp32_last': str(OUTPUT/'fp32_last.pt'),
    'qat_best': str(OUTPUT/'qat_best.pt'), 'qat_last': str(OUTPUT/'qat_last.pt'),
    'int8_model': str(OUTPUT/'selective_int8.pt'),
    'evaluation_json': str(OUTPUT/'evaluation.json'),
    'benchmark_json': str(OUTPUT/'benchmark.json'),
    'epoch_benchmarks': str(OUTPUT/'epoch_benchmarks.json'),
}
RUNTIME_CONFIG = Path('/kaggle/working/seadronessee_kaggle.yaml')
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False))
print(RUNTIME_CONFIG.read_text())

## 5. Train đúng epoch tiếp theo
FP32 phải hoàn tất 30/30 trước khi đổi `STAGE='qat'`. Mỗi lần chạy cell này chỉ chạy một epoch rồi thoát.

In [ ]:
def run_and_log(command, log_path):
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(command), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in process.stdout:
            print(line, end='', flush=True); log_file.write(line); log_file.flush()
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

assert torch.cuda.is_available(), 'Dataset đã sẵn sàng; hãy bật GPU trong Kaggle Settings trước khi train'
command = [sys.executable, '-u', 'scripts/train_next_epoch.py', '--config', str(RUNTIME_CONFIG), '--stage', STAGE]
if STAGE == 'qat': command += ['--variant', VARIANT]
run_and_log(command, OUTPUT/f'{STAGE}_train.log')

## 6. Kiểm tra rồi lưu checkpoint thành Kaggle Dataset version
Không đóng phiên trước khi cell upload hoàn tất. Nếu checkpoint dataset chưa tồn tại, KaggleHub tự tạo dataset private; nếu đã có, nó tạo version mới.

In [ ]:
def checkpoint_epoch(path):
    path = Path(path)
    return int(torch.load(path, map_location='cpu', weights_only=False).get('epoch', 0)) if path.exists() else 0

fp32_epoch = checkpoint_epoch(OUTPUT/'fp32_last.pt')
qat_epoch = checkpoint_epoch(OUTPUT/'qat_last.pt')
print(f'FP32={fp32_epoch}/30 QAT={qat_epoch}/11')
for path in sorted(OUTPUT.glob('*')):
    print(f'{path.name:30s} {path.stat().st_size/2**20:9.2f} MB')
kagglehub.dataset_upload(
    CHECKPOINT_DATASET, str(OUTPUT),
    version_notes=f'FP32 epoch {fp32_epoch}, QAT epoch {qat_epoch}',
)
print('Checkpoint dataset uploaded:', CHECKPOINT_DATASET)